# Analysis
Analyze a dataset in terms of
- clique distribution per split
- YouTube metadata (optional)


## DVI2
The cleaned version of *Discogs-VI-YT*. This dataset includes a few cliques and versions less than the first iteration of the dataset.

In [1]:
import os

# DVI2 (cleaned Discogs-VI-YT)
dvi2_dir = os.path.join("data", "dvi2", "dataset")
dvi2_path = os.path.join(dvi2_dir, "dvi_cleaned.jsonl")


In [2]:
import json

def read_json_lines(path: str):
    with open(path, "r") as f:
        data = [json.loads(line) for line in f]
    return data

dvi2_full = read_json_lines(dvi2_path)


In [3]:
def read_json(path: str):
    with open(path, "r") as f:
        data = json.load(f)
    return data

splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2_dir, f"{split}.json"))
    

In [4]:
import numpy as np

def clique_version_statistics(data):
    """
    Calculate statistics for the number of versions per clique.
    
    Args:
        data (dict): Dictionary containing cliques and their versions.
        
    Returns:
        dict: Statistics including min, max, median, mean, and std of versions per clique.
    """
    total = 0
    n_per_c = []
    print("===================================")
    print(f"{split} set:")
    for clique_id, versions in data[split].items():
        
        n = len(versions)
        total += n
        n_per_c.append(n)

    n_per_c = np.array(n_per_c)

    print(f"{'Cliques:':>20} {len(n_per_c):>10,}")
    print(f"{'Versions:':>20} {total:>10,}")
    print()
    print("Versions per clique:")
    print(f"{'min:':>20} {n_per_c.min():>10}")
    print(f"{'max:':>20} {n_per_c.max():>10}")
    print(f"{'median:':>20} {np.median(n_per_c):>10}")
    print(f"{'mean:':>20} {n_per_c.mean():>10.2f}")
    print(f"{'std:':>20} {n_per_c.std():>10.2f}")
    
for split in splits:
    clique_version_statistics(data)
        

train set:
            Cliques:     74,574
           Versions:    311,804

Versions per clique:
                min:          2
                max:        455
             median:        2.0
               mean:       4.18
                std:       8.40
val set:
            Cliques:      8,294
           Versions:     34,098

Versions per clique:
                min:          2
                max:        258
             median:        2.0
               mean:       4.11
                std:       8.01
test set:
            Cliques:     10,273
           Versions:    111,304

Versions per clique:
                min:          2
                max:        629
             median:        3.0
               mean:      10.83
                std:      27.85


## DVI2-FM
The cleaned and extended dataset. This dataset contains a few less cliques, but drastically more versions

In [5]:

# DVI2-FM (cleaned & extended Discogs-VI-YT)
dvi2fm_dir = os.path.join(dvi2_dir, "matched")
dvi2fm_path = os.path.join(dvi2fm_dir, "dvi_fm_filtered.jsonl")


In [6]:
splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2fm_dir, f"{split}.json"))

for split in splits:
    clique_version_statistics(data)
  

train set:
            Cliques:     76,740
           Versions:  1,011,398

Versions per clique:
                min:          2
                max:        483
             median:        5.0
               mean:      13.18
                std:      19.03
val set:
            Cliques:      8,529
           Versions:    110,490

Versions per clique:
                min:          2
                max:        258
             median:        5.0
               mean:      12.95
                std:      18.65
test set:
            Cliques:     10,519
           Versions:    221,480

Versions per clique:
                min:          2
                max:        629
             median:        7.0
               mean:      21.06
                std:      33.71


## YouTube metadata

In [ ]:
import pandas as pd

def read_dataset_split(data: dict):
    rows = []
    for clique_id, versions in data.items():
        for item in versions:
            item_with_clique = {"clique_id": clique_id, **item}
            rows.append(item_with_clique)
    return pd.DataFrame(rows)

# collect DVI2-FM splits into dataframe
df = pd.DataFrame()
for split in splits:
    df_tmp = read_dataset_split(data[split])
    df_tmp["split"] = split
    df = pd.concat([df, df_tmp], ignore_index=True)


In [ ]:
import pandas as pd

# join YouTube metadata with DVI2-FM
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(df.youtube_id),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    df,
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


In [ ]:
import pandas as pd

# join YouTube metadata with DVI2-FM
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')

print("JOIN...")
metadata = pd.merge(
    df,
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)

def time_to_seconds(time_str):
    if not isinstance(time_str, str):
        # doing nothing if the time_str is not a string
        return time_str
    parts = list(map(int, time_str.split(":")))
    if len(parts) == 3:  # HH:MM:SS
        h, m, s = parts
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:  # MM:SS
        m, s = parts
        return m * 60 + s
    elif len(parts) == 1:  # SS
        return parts[0]
    else:
        raise ValueError(f"Unrecognized time format: {time_str}")

print("Pre-process time...")
metadata["duration_secs"] = metadata["duration"].apply(time_to_seconds)
metadata["duration_mins"] = metadata["duration_secs"] / 60
metadata_filtered = metadata.dropna(subset=["id"])

print("Pre-process description...")
def get_description_str(descriptionSnippet):
    if isinstance(descriptionSnippet, list):
        return " ".join([d["text"] for d in descriptionSnippet]).replace("\n", " ").replace("\r", " ")
    else:
        return descriptionSnippet
metadata_filtered.loc[:,"description"] = metadata_filtered.descriptionSnippet.apply(get_description_str)

print("Done.")


### Collect n-Grams

In [57]:
import unicodedata
from collections import Counter
from itertools import islice
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Make sure you've downloaded stopwords & tokenizer models:
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

# Combine stopwords from multiple languages
languages = ['english', 'german', 'french', 'spanish', 'italian', 'portuguese']
stop_words = set()
for lang in languages:
    stop_words.update(stopwords.words(lang))

def normalize_unicode(text):
    # Normalize fancy fonts and remove diacritics (if any)
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def ngrams(tokens, n):
    return zip(*(islice(tokens, i, None) for i in range(n)))

def clean_and_tokenize(text):
    # Normalize text
    if isinstance(text, str):    
        text = normalize_unicode(text)
        tokens = word_tokenize(text.lower())
        return [word for word in tokens if word.isalnum() and word not in stop_words]
    else:
        return []

# Assuming your dataframe is called df
for column in ['title', 'description']:
    print("====================================")
    print(f"Processing column: {column}")
    metadata_filtered.loc[:,column + '_tokenized'] = metadata_filtered[column].apply(clean_and_tokenize)
    for n in [1, 2, 3, 4]:
        gram_str = f"{n}gram"
        print(f"Creating {gram_str}")
        metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(
            lambda x: list(ngrams(x, n)))


Processing column: title
Creating 1gram
Creating 2gram
Creating 3gram
Creating 4gram
Processing column: description


/tmp/ipykernel_1314774/2560943582.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + '_tokenized'] = metadata_filtered[column].apply(clean_and_tokenize)


Creating 1gram


/tmp/ipykernel_1314774/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 2gram


/tmp/ipykernel_1314774/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 3gram


/tmp/ipykernel_1314774/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


Creating 4gram


/tmp/ipykernel_1314774/2560943582.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,column + f'_{gram_str}'] = metadata_filtered[column + '_tokenized'].apply(


### Subsets by use case


In [40]:
def describe_subset(mask):
    """
    Print the number of unique cliques and total versions per split
    for the subset of metadata_filtered where mask is True.
    """
    subset = metadata_filtered[mask]
    for split in subset['split'].unique():
        split_df = subset[subset['split'] == split]
        n_cliques = split_df['clique_id'].nunique()
        n_versions = len(split_df)
        print(f"{split.capitalize()} set: {n_cliques:,} cliques, {n_versions:,} versions")
        
        

#### *Covers*

In [99]:
for instrument in ["guitar", "piano", "drum", "bass"]:
    mask = metadata_filtered["title_2gram"].apply(lambda grams: (instrument, "cover",) in grams)
    print(f"Filtering for {instrument} covers...")
    describe_subset(mask)
    print("\n")


Filtering for guitar covers...
Train set: 1,752 cliques, 2,842 versions
Val set: 185 cliques, 293 versions
Test set: 329 cliques, 571 versions


Filtering for piano covers...
Train set: 784 cliques, 975 versions
Val set: 92 cliques, 116 versions
Test set: 160 cliques, 216 versions


Filtering for drum covers...
Train set: 1,187 cliques, 1,761 versions
Val set: 138 cliques, 222 versions
Test set: 210 cliques, 317 versions


Filtering for bass covers...
Train set: 1,547 cliques, 2,256 versions
Val set: 168 cliques, 249 versions
Test set: 277 cliques, 455 versions




#### Karaoke

In [ ]:
mask = metadata_filtered.duration_secs < t

print(f"Filtering out videos longer than {t} seconds...")
describe_subset(mask)


#### *Short Segments*

In [100]:
t = 20
mask = metadata_filtered["title_1gram"].apply(lambda grams: ("karaoke",) in grams)

print(f"Filtering for karaoke versions...")
describe_subset(mask)


Filtering for karaoke versions...
Train set: 6,861 cliques, 12,377 versions
Val set: 686 cliques, 1,295 versions
Test set: 1,168 cliques, 2,204 versions


#### *Reactions*


In [ ]:
mask = metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "reaction") in grams)
mask = mask | metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "hearing") in grams)
mask = mask | metadata_filtered["title_3gram"].apply(lambda grams: ("first", "time", "seeing") in grams)

print(f"Filtering for reaction videos...")
describe_subset(mask)


Filtering for reaction videos...
Train set: 1,339 cliques, 2,272 versions
Val set: 147 cliques, 267 versions
Test set: 333 cliques, 594 versions


#### *Instructional*

In [93]:
mask = mask | metadata_filtered["title_1gram"].apply(lambda grams: ("tutorial",) in grams)
mask = mask | metadata_filtered["title_1gram"].apply(lambda grams: ("instruction",) in grams)
mask = mask | metadata_filtered["title_1gram"].apply(lambda grams: ("lesson",) in grams)

print(f"Filtering for instructional videos...")
describe_subset(mask)


Filtering for instructional videos...
Train set: 3,207 cliques, 6,176 versions
Val set: 348 cliques, 617 versions
Test set: 656 cliques, 1,476 versions
